In [10]:
import os, sys
cur_directory = os.getcwd()
project_folder = os.path.abspath(os.path.join(cur_directory, os.pardir, os.pardir))
save_directory = os.path.join(cur_directory, 'saved')
sys.path.append(os.path.join(project_folder, 'analysis-suite'))

In [11]:
import numpy as np
import matplotlib as mpl
from matplotlib import pyplot as plt
import analyses as als
import datasets as ds
import models as mds
import tools
from random import sample
from tqdm import tqdm
from argparse import Namespace
from time import gmtime, strftime
import pickle

## Params

In [12]:
V_width = 1
# V_E_buffer_array = np.arange(16).astype(int)
V_E_buffer_array = np.linspace(0,30,16).astype(int)
# V_E_buffer_array = np.arange(30).astype(int)
E_width = 1

K = 8

N_training_repeats = 5
critic_params = {
    "hidden_dim": 32,
    "embed_dim": 8,
    "layers": 2,
    "activation": "tanh",
}
train_opt_params = {"track_quantities": False, "quiet": True, "iterations": 1}
opt_params = {"track_quantities": False, "quiet": True, "iterations": 1}
mi_measurement_params = {"quiet": True, "batch_size": 500, "N_batches": 500}

train_fraction = 0.5
samp_keep = 1

# shuffle time bins in a way that is the same across movie repeats
shuffle_time = False

# Put variable names here for parameters we want saved.
save_vars = {
    "train_opt_params",
    "mi_measurement_params",
    "N_training_repeats",
    "critic_params",
    "opt_params",
    "K",
    "V_width",
    "V_E_buffer_array",
    "E_width",
    "train_fraction",
    "samp_keep",
    "shuffle_time"
}


## Prepare data

In [13]:
data_path = os.path.join(project_folder, 'data', 'nat_scenes_data')
D = ds.DataNatScenes(data_path, mov_ind=4)
mov_ind = D.mov_ind
mov_name = D.movname
save_vars.add('mov_ind')

# data_path = os.path.join(project_folder, 'data', 'bar_stim_data')
# D = ds.DataBar(data_path)
# mov_name = 'bar'

save_vars.add('mov_name')
print(f'current stimulus is {mov_name}')

current stimulus is opticflow


## Run analysis

### Train coarse-grainers

In [14]:
experiment_dims = (len(V_E_buffer_array), N_training_repeats)
coarse_grainer_array = np.zeros(experiment_dims, dtype=object)
train_points = np.zeros(experiment_dims, dtype=object)
test_points = np.zeros(experiment_dims, dtype=object)

for i, dt in tqdm(enumerate(V_E_buffer_array)):
    for training_repeat in range(N_training_repeats):
        trial_inds = (i, training_repeat)

        # Domains, training and test subsets
        D.reset()
        D.trim(20)

        # shuffling stuff
        if shuffle_time:
            D.shuffle_time()

        # D.subset_points_uniform(time_order_inds)  # shuffle or preserve time ordering
        V, E = D.create_V_and_E_datasets(V_width, dt, E_width, samp_keep)
        (
            current_train_points,
            current_test_points,
        ) = tools.generate_train_test_indices(V.P, train_fraction)
        train_points[trial_inds] = np.array(current_train_points)
        test_points[trial_inds] = np.array(current_test_points)
 
        # train IB on training points
        coarse_grainer = mds.PolynomialLayer(output_dim=K, order=1)
        critic_function = mds.SeparableCritic(**critic_params)
        
        # Train current
        V.subset_points(current_train_points)
        E.subset_points(current_train_points)
        als.IB(
            X=V.X,
            Y=E.X,
            coarse_grainer=coarse_grainer,
            critic_function=critic_function,
            **train_opt_params
        )

        # store coarse-grainer
        coarse_grainer_array[trial_inds] = coarse_grainer

0it [00:00, ?it/s]

16it [07:08, 26.76s/it]


### Get MI with new critics

In [15]:
experiment_dims = (len(V_E_buffer_array), N_training_repeats)
mi_averages_test = np.zeros(experiment_dims)
mi_averages_train = np.zeros(experiment_dims)
mi_stdevs_train = np.zeros(experiment_dims)
mi_stdevs_test = np.zeros(experiment_dims)

save_vars.update(
    {
        "mi_averages_train",
        "mi_averages_test",
        "mi_stdevs_train",
        "mi_stdevs_test",
        "train_points",
        "test_points",
    }
)

for i, V_E_buffer in tqdm(enumerate(V_E_buffer_array)):
    for training_repeat in range(N_training_repeats):
        
        trial_inds = (i, training_repeat)

        # Domains, training and test subsets
        D.reset()
        D.trim(20)
        if shuffle_time:
            D.shuffle_time_and_trial()
        V, E = D.create_V_and_E_datasets(V_width, V_E_buffer, E_width, samp_keep)
        current_train_points = train_points[trial_inds]
        current_test_points = test_points[trial_inds]


        # get coarse grainer and fresh critic function
        coarse_grainer = coarse_grainer_array[trial_inds]
        critic_function = mds.SeparableCritic(**critic_params)

        # train new critic function
        V.subset_points(current_train_points)
        E.subset_points(current_train_points)
        T = coarse_grainer(V.X.T).numpy().T
        als.train_NCE(T, E.X, critic_function, **opt_params)

        # Measure InfoNCE for train
        mi_train = als.MI_NCE(
            X=T, Y=E.X, critic_function=critic_function, **mi_measurement_params
        )
        mi_averages_train[trial_inds] = mi_train["mi"]
        mi_stdevs_train[trial_inds] = mi_train["mi_stdev"]

        # Measure InfoNCE for test
        V.reset()
        E.reset()
        V.subset_points(current_test_points)
        E.subset_points(current_test_points)
        T = T = coarse_grainer(V.X.T).numpy().T
        mi_test = als.MI_NCE(
            X=T, Y=E.X, critic_function=critic_function, **mi_measurement_params
        )
        mi_averages_test[trial_inds] = mi_test["mi"]
        mi_stdevs_test[trial_inds] = mi_test["mi_stdev"]

16it [11:37, 43.60s/it]


## Save

In [16]:
# select variables
coarse_grainer_type = type(coarse_grainer)
critic_type = type(critic_function)
save_vars.update({"critic_type", "coarse_grainer_type"})
local_dict = locals()
output_data = {k: local_dict[k] for k in save_vars}

In [17]:
# save things
timestamp = strftime("%Y.%m.%d %Hh %Mm %Ss.dat",gmtime())
# full_file_path = os.path.join(save_directory, strftime("%Y.%m.%d %Hh %Mm %Ss",gmtime())+'.dat')
if shuffle_time:
    prepend = "(shuffled) "
else:
    prepend = ""

stim = mov_name
full_file_path = os.path.join(save_directory, stim, prepend + f'longer training bound.dat')
print(f"data will be saved to:\n{full_file_path}")

data will be saved to:
/home/adam/Documents/projects/retina-past-future-ib/analyses/multiplexed prediction/saved/opticflow/longer training bound.dat


In [9]:
os.makedirs(os.path.dirname(full_file_path), exist_ok=True)
with open(full_file_path, 'wb+') as f:
            pickle.dump(output_data, f)